# 5-2: LiP-MS Structure Mapping

Maps significant LiP-MS peptides and metabolite-correlating peptides onto a 3D ribosome structure (PDB: **8Z70**).

**Sources:**
- LiP-MS hits: `results/diffStructurome_G/N/P.tsv`
- Correlations: `source_data/LiPMS_vs_Metab.tsv`
- Structure: RCSB 8Z70 (*S. cerevisiae* 80S ribosome)


In [1]:
import pandas as pd
import numpy as np
import requests
import os
import pickle
import py3Dmol
import freesasa
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display
from plotnine import *

print(f'py3Dmol version: {py3Dmol.__version__}')
print(f'Working directory: {os.getcwd()}')

py3Dmol version: 2.5.4
Working directory: /Users/xhernandez/Documents/pmi_translation


## Parameters

Edit the cell below to re-run this analysis for a different protein complex. All downstream cells adapt automatically.

In [2]:
# ============================================================
# PARAMETERS — edit only this cell to switch to another structure
# ============================================================

PDB_ID         = '8Z70'                       # RCSB PDB identifier
ANALYSIS_LABEL = 'S. cerevisiae 80S ribosome' # label used in plot titles
OUTPUT_PREFIX  = 'ribosome'                   # prefix for all output files

# Restrict to a subset of UniProt IDs present in both the LiP-MS data and the PDB.
# None = use ALL proteins found in the intersection automatically.
# Example: UNIPROT_FILTER = ['P04641', 'P04643']
UNIPROT_FILTER = None

# Color palette (used throughout downstream cells)
COLOR_UNDETECTED = '#888888'  # other / unclassified protein chains
COLOR_BASE       = "#888888"  # fallback grey
COLOR_RPL        = '#8fddc0'  # large ribosomal subunit (RPL) background
COLOR_RPS        = "#4c7b6a"  # small ribosomal subunit (RPS) background
COLOR_SIG        = "#ea9c8e"  # significant LiP peptides, NOT metabolite-correlated
COLOR_CORR       = '#d8624d'  # significant LiP peptides, correlating with metabolites


## 1. Load LiP-MS Data

In [3]:
# Load differential structurome results from all three nutrient conditions
diff_g = pd.read_csv('results/diffStructurome_G.tsv', sep='\t', index_col='quantID')
diff_n = pd.read_csv('results/diffStructurome_N.tsv', sep='\t', index_col='quantID')
diff_p = pd.read_csv('results/diffStructurome_P.tsv', sep='\t', index_col='quantID')

# Build annotprot: union of all conditions, deduplicated by quantID
annot_cols = ['Peptide', 'Protein', 'gene_name', 'startPosition', 'endPosition', 'isTryptic']
annotprot = pd.concat([diff_g[annot_cols], diff_n[annot_cols], diff_p[annot_cols]])
annotprot = annotprot[~annotprot.index.duplicated(keep='first')]

# siglip: peptides significant (adj.P.Val < 0.05) in at least one condition
siglip_g = set(diff_g.index[(diff_g['adj.P.Val'] < 0.05) & diff_g['adj.P.Val'].notna()])
siglip_n = set(diff_n.index[(diff_n['adj.P.Val'] < 0.05) & diff_n['adj.P.Val'].notna()])
siglip_p = set(diff_p.index[(diff_p['adj.P.Val'] < 0.05) & diff_p['adj.P.Val'].notna()])
siglip = list(siglip_g | siglip_n | siglip_p)

print(f'Total peptides in annotprot: {len(annotprot)}')
print(f'Significant peptides (siglip): {len(siglip)}')
print(f'  G: {len(siglip_g)}, N: {len(siglip_n)}, P: {len(siglip_p)}')
annotprot.head(3)

Total peptides in annotprot: 29732
Significant peptides (siglip): 5540
  G: 3204, N: 1112, P: 1782


,Peptide,Protein,gene_name,startPosition,endPosition,isTryptic
quantID,,,,,,
EMIISSLSPLEPHSQNEILQYYL,EMIISSLSPLEPHSQNEILQYYL,P12945,NAT1,832,854,Specific
SFDDLHK,SFDDLHK,P17709,GLK1,2,8,Specific-N
EQNEDLGLSDIKVE,EQNEDLGLSDIKVE,P53303,ZPR1,473,486,Specific


## 2. Load Correlation Data

In [4]:
# Load LiP-MS vs Metabolomics correlation results
# X1 = peptide (quantID), X2 = metabolite, lfdr = local false discovery rate
cors = pd.read_csv('source_data/LiPMS_vs_Metab.tsv', sep='\t', index_col=0)
print(f'Columns: {list(cors.columns)}')
print(f'Total peptide-metabolite pairs tested: {len(cors)}')

# Correlating peptides: those with lfdr < 0.05 in at least one metabolite pair
correlating_pepts = set(cors.loc[cors['lfdr'] < 0.05, 'X1'])
print(f'Peptides with significant metabolite correlations (lfdr < 0.05): {len(correlating_pepts)}')

# Classify siglip peptides into two categories:
#   sig_only     → significant LiP but NOT metabolite-correlated → color #4c7b6a
#   sig_and_corr → significant LiP AND metabolite-correlated   → color #d8624d
sig_only     = [p for p in siglip if p not in correlating_pepts]
sig_and_corr = [p for p in siglip if p in correlating_pepts]

print(f'\nSignificant, non-correlating (→ #4c7b6a): {len(sig_only)}')
print(f'Significant AND correlating  (→ #d8624d): {len(sig_and_corr)}')

Columns: ['X1', 'X2', 'pval', 'qval', 'lfdr', 'R2', 'annot_pept', 'annot_metab', 'GO_translation']
Total peptide-metabolite pairs tested: 1429866
Peptides with significant metabolite correlations (lfdr < 0.05): 1558

Significant, non-correlating (→ #4c7b6a): 3982
Significant AND correlating  (→ #d8624d): 1558


## 3. Fetch Ribosome Structure and SIFTS Mapping

In [5]:
os.makedirs('data', exist_ok=True)
cif_path = f'data/{PDB_ID}.cif'

if not os.path.exists(cif_path):
    print(f'Downloading {PDB_ID} from RCSB (may take a minute for large structures)...')
    url = f'https://files.rcsb.org/download/{PDB_ID}.cif'
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    with open(cif_path, 'w') as f:
        f.write(resp.text)
    print(f'Saved to {cif_path}')
else:
    print(f'Using cached {cif_path}')

file_size_mb = os.path.getsize(cif_path) / 1e6
print(f'File size: {file_size_mb:.1f} MB')

Using cached data/8Z70.cif
File size: 22.8 MB


In [6]:
def get_sifts_mapping(pdb_id):
    """
    Fetch UniProt -> PDB chain/residue mapping from PDBe SIFTS API.
    Returns:
        sifts:      {uniprot_id: [{'chain_id', 'unp_start', 'unp_end', 'pdb_start', 'pdb_end'}]}
        sifts_ident: {uniprot_id: identifier_string}  e.g. 'RL2A_YEAST'
    """
    url = f'https://www.ebi.ac.uk/pdbe/api/mappings/uniprot/{pdb_id.lower()}'
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    pdb_key = pdb_id.lower()

    sifts       = {}
    sifts_ident = {}
    for uniprot_id, info in data[pdb_key]['UniProt'].items():
        chain_maps = []
        for m in info['mappings']:
            chain_maps.append({
                'chain_id': m['chain_id'],
                'unp_start': m['unp_start'],
                'unp_end':   m['unp_end'],
                'pdb_start': m['start']['residue_number'],
                'pdb_end':   m['end']['residue_number'],
            })
        sifts[uniprot_id]       = chain_maps
        sifts_ident[uniprot_id] = info.get('identifier', '')
    return sifts, sifts_ident

print(f'Fetching SIFTS mapping for {PDB_ID}...')
sifts, sifts_ident = get_sifts_mapping(PDB_ID)
print(f'Proteins with SIFTS mappings in {PDB_ID}: {len(sifts)}')
print(f'Example UniProt IDs: {list(sifts.keys())[:8]}')


Fetching SIFTS mapping for 8Z70...
Proteins with SIFTS mappings in 8Z70: 75
Example UniProt IDs: ['P0CX45', 'P14126', 'P10664', 'P26321', 'P05739', 'P05737', 'P17076', 'P05738']


## 4. Identify Ribosomal Peptides

In [7]:
# Find which LiP-MS proteins overlap with proteins in the ribosome structure
# Handle comma-separated protein IDs (ambiguously assigned peptides)
def _proteins_of(protein_str):
    """Return list of individual UniProt IDs from a possibly comma-separated string."""
    return [p.strip() for p in str(protein_str).split(',')]

lipm_proteins     = {p for prot_str in annotprot['Protein'] for p in _proteins_of(prot_str)}
ribosome_proteins = set(sifts.keys())
matched_proteins  = lipm_proteins & ribosome_proteins

# Apply optional UniProt filter (set UNIPROT_FILTER in the Parameters cell)
if UNIPROT_FILTER is not None:
    matched_proteins = matched_proteins & set(UNIPROT_FILTER)
    print(f'Restricted to {len(UNIPROT_FILTER)} specified UniProt IDs \u2192 {len(matched_proteins)} matched')

print(f'Proteins in LiP-MS dataset:  {len(lipm_proteins)}')
print(f'Proteins in {PDB_ID}:         {len(ribosome_proteins)}')
print(f'Matched proteins:            {len(matched_proteins)}')

# Filter peptide lists to matched proteins only
def filter_to_target(pept_list):
    return [p for p in pept_list
            if p in annotprot.index and
            any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]

siglip_ribo       = filter_to_target(siglip)
sig_only_ribo     = filter_to_target(sig_only)
sig_and_corr_ribo = filter_to_target(sig_and_corr)

print(f'\nPeptides from matched proteins in siglip:      {len(siglip_ribo)}')
print(f'  Sig, non-correlating (\u2192 {COLOR_SIG}):  {len(sig_only_ribo)}')
print(f'  Sig AND correlating  (\u2192 {COLOR_CORR}):  {len(sig_and_corr_ribo)}')

# All peptides from matched proteins (background for SASA analysis)
all_ribo_pepts = [p for p in annotprot.index
                  if any(prot in matched_proteins for prot in _proteins_of(annotprot.loc[p, 'Protein']))]
print(f'\nAll matched-protein peptides (SASA background): {len(all_ribo_pepts)}')

# Show matched proteins and gene names
ribo_annot = (annotprot[annotprot['Protein'].apply(
                  lambda x: any(p in matched_proteins for p in _proteins_of(x)))]
              [['Protein', 'gene_name']].drop_duplicates().sort_values('gene_name'))
print(f'\nMatched proteins ({len(ribo_annot)}):')
display(ribo_annot)


Proteins in LiP-MS dataset:  2510
Proteins in 8Z70:         75
Matched proteins:            44

Peptides from matched proteins in siglip:      127
  Sig, non-correlating (→ #ea9c8e):  85
  Sig AND correlating  (→ #d8624d):  42

All matched-protein peptides (SASA background): 703

Matched proteins (44):


,Protein,gene_name
quantID,,
AAEPHAVSL,P38011,ASC1
WGFTNLDRPEYLKK,P41805,RPL10
WAAAAVCEK,P36105,RPL14A
YNWICDPVHK,P05748,RPL15A
VFEGIPPPYDKK,P26784,RPL16A
YESSPSHIELVVTEKEEAVAK,P05740,RPL17A
AQGVAVQLK,Q02753,RPL21A
TFTVDVSSPTENGVFDPASYAK,P05749,RPL22A
MKVEIDSF,P04449,RPL24A


## 5. Map Peptide Positions to PDB Residues

Uses SIFTS to convert UniProt sequence positions (startPosition, endPosition from LiP-MS) to PDB chain + residue numbers.

In [8]:
from collections import defaultdict

def map_peptide_to_pdb(pept_start, pept_end, chain_maps):
    """
    Map UniProt peptide [pept_start, pept_end] to PDB chain + residue list.
    Handles cases where a peptide spans multiple mapping segments.
    Returns: [{'chain': chain_id, 'residues': [res1, res2, ...]}]
    """
    results = []
    for cm in chain_maps:
        ov_start = max(pept_start, cm['unp_start'])
        ov_end   = min(pept_end,   cm['unp_end'])
        if ov_start <= ov_end:
            offset  = cm['pdb_start'] - cm['unp_start']
            pdb_res = list(range(ov_start + offset, ov_end + offset + 1))
            results.append({'chain': cm['chain_id'], 'residues': pdb_res})
    return results

def map_pept_list(pept_list):
    """Map a list of peptide IDs to PDB chain + residue positions.
    Handles peptides ambiguously assigned to multiple comma-separated proteins:
    tries each protein with its corresponding start/end position in order,
    and uses the first one that maps successfully to the PDB structure."""
    result = {}
    for pept_id in pept_list:
        if pept_id not in annotprot.index:
            continue
        row      = annotprot.loc[pept_id]
        proteins = _proteins_of(row['Protein'])
        starts   = [s.strip() for s in str(row['startPosition']).split(',')]
        ends     = [e.strip() for e in str(row['endPosition']).split(',')]
        for i, uniprot_id in enumerate(proteins):
            if uniprot_id not in sifts:
                continue
            try:
                pept_start = int(starts[i] if i < len(starts) else starts[0])
                pept_end   = int(ends[i]   if i < len(ends)   else ends[0])
            except (ValueError, TypeError, IndexError):
                continue
            pdb_positions = map_peptide_to_pdb(pept_start, pept_end, sifts[uniprot_id])
            if pdb_positions:
                result[pept_id] = pdb_positions
                break  # use first successful mapping
    return result

# Map significant peptides (used for visualization coloring)
pept_to_pdb      = map_pept_list(siglip_ribo)
pdb_sig_only     = {p: pept_to_pdb[p] for p in sig_only_ribo     if p in pept_to_pdb}
pdb_sig_corr     = {p: pept_to_pdb[p] for p in sig_and_corr_ribo if p in pept_to_pdb}

# Map ALL matched-protein peptides (needed for SASA background comparison)
all_ribo_pept_to_pdb = map_pept_list(all_ribo_pepts)

print(f'Significant peptides mapped to PDB:              {len(pept_to_pdb)}')
print(f'  Non-correlating (\u2192 {COLOR_SIG}):       {len(pdb_sig_only)}')
print(f'  Correlating     (\u2192 {COLOR_CORR}):       {len(pdb_sig_corr)}')
print(f'All matched-protein peptides mapped:             {len(all_ribo_pept_to_pdb)}')

# ------------------------------------------------------------------
# Build per-residue group sets (used for visualization and SASA)
# ------------------------------------------------------------------
def residues_from_pept_dict(pept_dict):
    """Collect all (chain, res_num) pairs covered by any peptide in the dict."""
    res_set = set()
    for positions in pept_dict.values():
        for pos in positions:
            for rn in pos['residues']:
                res_set.add((pos['chain'], rn))
    return res_set

detected_res_set = residues_from_pept_dict(all_ribo_pept_to_pdb)   # any detected peptide
sig_res_set      = residues_from_pept_dict(pept_to_pdb)             # any siglip peptide
corr_res_set     = residues_from_pept_dict(pdb_sig_corr)            # correlating peptide

# All PDB residues for matched proteins (from SIFTS) -- defines full expected coverage
sifts_residues = set()
for uniprot_id in matched_proteins:
    if uniprot_id not in sifts:
        continue
    for cm in sifts[uniprot_id]:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            sifts_residues.add((cm['chain_id'], unp_res + offset))

print(f'\nPDB residues in matched proteins (SIFTS):  {len(sifts_residues)}')
print(f'  Covered by any detected peptide:          {len(detected_res_set)}')
print(f'  Covered by significant peptide:           {len(sig_res_set)}')
print(f'  Covered by correlating peptide:           {len(corr_res_set)}')
print(f'  Not detected (no LiP-MS coverage):        {len(sifts_residues - detected_res_set)}')


Significant peptides mapped to PDB:              123
  Non-correlating (→ #ea9c8e):       82
  Correlating     (→ #d8624d):       41
All matched-protein peptides mapped:             673

PDB residues in matched proteins (SIFTS):  7100
  Covered by any detected peptide:          3736
  Covered by significant peptide:           1303
  Covered by correlating peptide:           453
  Not detected (no LiP-MS coverage):        3364


## 6. Solvent Accessibility Analysis

Computes per-residue **relative SASA (rSASA)** on the full complex using freesasa, then compares rSASA distributions across four groups (one data point = one residue, assigned to highest-priority group):

Results are cached to `data/<PDB_ID>_residue_sasa.pkl` after the first run.

In [9]:
sasa_cache_path = f'data/{PDB_ID}_residue_sasa.pkl'

if os.path.exists(sasa_cache_path):
    print(f'Loading cached SASA from {sasa_cache_path}...')
    with open(sasa_cache_path, 'rb') as f:
        res_sasa = pickle.load(f)
    print(f'Loaded rSASA for {len(res_sasa)} residues.')
else:
    import gemmi

    # ── 1. Parse CIF with gemmi (fast C++ library, seconds not hours) ──────────
    print(f'Parsing {PDB_ID} CIF with gemmi...')
    st    = gemmi.read_structure(cif_path)
    model = st[0]

    # ── 2. PDB format only allows 1-char chain IDs.  Remap any multi-char     ──
    #       auth_asym_ids and keep a reverse lookup so we can restore them later.
    single_used = {ch.name for ch in model if len(ch.name) == 1}
    spare       = [c for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789'
                   if c not in single_used]
    pdb_to_auth = {ch.name: ch.name for ch in model if len(ch.name) == 1}
    spare_idx   = 0
    for chain in model:
        if len(chain.name) > 1:
            orig          = chain.name
            new_id        = spare[spare_idx] if spare_idx < len(spare) else '?'
            spare_idx    += 1
            pdb_to_auth[new_id] = orig
            chain.name    = new_id   # rename in-place for PDB output

    print(f'  {len(list(model))} chains; {spare_idx} multi-char chain IDs remapped for PDB format')

    # ── 3. Write temporary PDB and run freesasa ────────────────────────────────
    tmp_pdb = cif_path.replace('.cif', '_freesasa_tmp.pdb')
    st.write_pdb(tmp_pdb)
    print(f'  Temporary PDB written ({os.path.getsize(tmp_pdb)/1e6:.0f} MB)')

    print('Computing SASA with freesasa...')
    try:
        fs_structure = freesasa.Structure(tmp_pdb)
        sasa_result  = freesasa.calc(fs_structure)
    finally:
        os.remove(tmp_pdb)

    # ── 4. Build per-residue rSASA lookup with original auth_asym_id labels ───
    res_sasa = {}
    for pdb_chain, chain_areas in sasa_result.residueAreas().items():
        auth_chain = pdb_to_auth.get(pdb_chain, pdb_chain)
        for res_label, res_area in chain_areas.items():
            try:
                res_num = int(str(res_label).strip())
                rsa     = res_area.relativeTotal
                if rsa is not None:
                    res_sasa[(auth_chain, res_num)] = float(rsa)
            except (ValueError, TypeError):
                pass

    with open(sasa_cache_path, 'wb') as f:
        pickle.dump(res_sasa, f)
    print(f'SASA cached to {sasa_cache_path}')

n_valid = sum(1 for v in res_sasa.values() if not np.isnan(v))
print(f'Residues with valid rSASA: {n_valid}')


Loading cached SASA from data/8Z70_residue_sasa.pkl...
Loaded rSASA for 14127 residues.
Residues with valid rSASA: 8739


In [10]:
def assign_residue_group(res_key):
    if res_key in corr_res_set:
        return 'correlating'
    elif res_key in sig_res_set:
        return 'significant'
    elif res_key in detected_res_set:
        return 'detected, non-sig'
    else:
        return 'not detected'

# Build per-residue rSASA table
sasa_rows = []
for res_key in sifts_residues:
    if res_key not in res_sasa:
        continue
    rsa = res_sasa[res_key]
    if np.isnan(rsa):
        continue
    sasa_rows.append({'chain': res_key[0], 'res_num': res_key[1],
                      'lip_group': assign_residue_group(res_key), 'rSASA': rsa})

sasa_df = pd.DataFrame(sasa_rows)

group_order  = ['not detected', 'detected, non-sig', 'significant', 'correlating']
group_labels = ['Not\ndetected', 'Detected,\nnon-sig', 'Significant\n(all)', 'Correlating\nw/ metabolites']
group_colors = {
    'not detected':      COLOR_UNDETECTED,
    'detected, non-sig': COLOR_BASE,
    'significant':       COLOR_SIG,
    'correlating':       COLOR_CORR,
}

sasa_df['lip_group'] = pd.Categorical(sasa_df['lip_group'], categories=group_order, ordered=True)

print('Group sizes (residues):')
print(sasa_df['lip_group'].value_counts().reindex(group_order).to_string())

# ── Statistical tests (one-tailed MWU: fg > ref) ─────────────────────────────
def mwu_test(fg_vals, ref_vals):
    _, pval = stats.mannwhitneyu(fg_vals, ref_vals, alternative='greater')
    stars = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    return stars, pval

def grp_vals(grp):
    return sasa_df.loc[sasa_df['lip_group'] == grp, 'rSASA'].values

nd  = grp_vals('not detected')
dns = grp_vals('detected, non-sig')

# Set 1: all groups vs not detected; Set 2: sig/corr vs detected non-sig
test_pairs = [
    ('not detected',      'detected, non-sig', nd),
    ('not detected',      'significant',       nd),
    ('not detected',      'correlating',       nd),
    ('detected, non-sig', 'significant',       dns),
    ('detected, non-sig', 'correlating',       dns),
]
tests = []
for ref_grp, fg_grp, ref_vals in test_pairs:
    fg_vals = grp_vals(fg_grp)
    if len(fg_vals) >= 3 and len(ref_vals) >= 3:
        stars, pval = mwu_test(fg_vals, ref_vals)
        tests.append({'ref': ref_grp, 'fg': fg_grp, 'stars': stars,
                      'pval': pval})

# ── Bracket annotations ───────────────────────────────────────────────────────
y_max  = float(sasa_df['rSASA'].max())
y_step = y_max * 0.09
x_pos  = {g: i + 1 for i, g in enumerate(group_order)}
tick_h = y_step * 0.25

ref_bases = {'not detected': y_max * 1.05, 'detected, non-sig': y_max * 1.42}
ref_k     = {'not detected': 0,            'detected, non-sig': 0}

segs, txts = [], []
for t in tests:
    x1   = x_pos[t['ref']]
    x2   = x_pos[t['fg']]
    k    = ref_k[t['ref']]
    y_br = ref_bases[t['ref']] + k * y_step
    ref_k[t['ref']] += 1
    segs += [
        {'x': x1, 'xend': x2, 'y': y_br,          'yend': y_br},
        {'x': x1, 'xend': x1, 'y': y_br - tick_h, 'yend': y_br},
        {'x': x2, 'xend': x2, 'y': y_br - tick_h, 'yend': y_br},
    ]
    txts.append({
        'x':     (x1 + x2) / 2,
        'y':     y_br + y_step * 0.08,
        'label': f'{t["stars"]} p={t["pval"]:.1e}',
    })

seg_df = pd.DataFrame(segs)
txt_df = pd.DataFrame(txts)
y_lim  = ref_bases['detected, non-sig'] + ref_k['detected, non-sig'] * y_step + y_step

# ── plotnine figure ───────────────────────────────────────────────────────────
p = (
    ggplot(sasa_df, aes(x='lip_group', y='rSASA', fill='lip_group'))
    + geom_boxplot(outlier_shape='', width=0.5, show_legend=False)
    + geom_jitter(width=0.2, alpha=0.2, size=0.5,
                  color='#333333', show_legend=False)
    + geom_segment(data=seg_df,
                   mapping=aes(x='x', xend='xend', y='y', yend='yend'),
                   inherit_aes=False, size=0.5, color='black')
    + geom_text(data=txt_df,
                mapping=aes(x='x', y='y', label='label'),
                inherit_aes=False, size=6, ha='center', va='bottom')
    + scale_x_discrete(limits=group_order, labels=group_labels)
    + scale_fill_manual(values=group_colors)
    + coord_cartesian(ylim=(None, y_lim))
    + labs(x='', y='Relative SASA per residue')
    + theme_classic()
    + theme(axis_text_x=element_text(size=9))
)

os.makedirs('figure_panels', exist_ok=True)
p.save(f'figure_panels/{OUTPUT_PREFIX}_sasa_boxplot.pdf', bbox_inches='tight')
p.draw()
print(f'Saved: figure_panels/{OUTPUT_PREFIX}_sasa_boxplot.pdf')


Group sizes (residues):
lip_group
not detected         2238
detected, non-sig    1738
significant           695
correlating           371


/Users/xhernandez/miniconda3/envs/lipms_structure/lib/python3.11/site-packages/plotnine/ggplot.py:623: PlotnineWarning: Saving 6.4 x 4.8 in image.
/Users/xhernandez/miniconda3/envs/lipms_structure/lib/python3.11/site-packages/plotnine/ggplot.py:624: PlotnineWarning: Filename: figure_panels/ribosome_sasa_boxplot.pdf


Saved: figure_panels/ribosome_sasa_boxplot.pdf


## 7. Visualization: Peptide Level

All atoms shown as spacefill (grey). Significant peptide residues highlighted by category.

In [11]:
# Load structure string (done once, reused for both views)
print(f'Loading CIF structure ({file_size_mb:.1f} MB)...')
with open(cif_path) as f:
    cif_str = f.read()
print('Done.')

Loading CIF structure (22.8 MB)...
Done.


In [12]:
import re as _re

def _get_trna_chains(cif_text):
    '''Parse tRNA auth_asym_id chain IDs from mmCIF metadata + atom_site records.
    Step 1: find tRNA entity IDs from _entity table (label_asym_id based).
    Step 2: find matching label_asym_id from _struct_asym blocks.
    Step 3: map label_asym_id -> auth_asym_id via a scan of _atom_site records,
            because py3Dmol (and SIFTS) use auth_asym_id for chain selection.
    '''
    # ── Step 1: find tRNA entity IDs ────────────────────────────────────────
    cut = cif_text.find('\n_atom_site.')
    if cut < 0:
        cut = cif_text.find('\nloop_\n_atom_site.')
    meta = cif_text[:cut] if cut > 0 else cif_text[:500_000]

    trna_eids = set()
    for m in _re.finditer(r'^\s*(\d+)\s+\S+\s+\S+\s+tRNA\b',
                          meta, _re.MULTILINE | _re.IGNORECASE):
        trna_eids.add(m.group(1))
    for m in _re.finditer(
            r"^\s*(\d+)\s+\S+\s+\S+\s+'([^']*(?:tRNA|transfer\s+RNA)[^']*)'",
            meta, _re.MULTILINE | _re.IGNORECASE):
        trna_eids.add(m.group(1))
    for m in _re.finditer(
            r'^\s*(\d+)\s+\S+\s+\S+\s+"([^"]*(?:tRNA|transfer\s+RNA)[^"]*)"',
            meta, _re.MULTILINE | _re.IGNORECASE):
        trna_eids.add(m.group(1))
    print(f'  tRNA entity IDs: {trna_eids}')
    if not trna_eids:
        return set()

    # ── Step 2: find label_asym_id for tRNA entities ─────────────────────────
    label_ids = set()
    search_pos = 0
    while True:
        sa_idx = meta.find('_struct_asym.id', search_pos)
        if sa_idx < 0:
            break
        block_start = meta.rfind('loop_', 0, sa_idx)
        block_end   = meta.find('\nloop_', sa_idx)
        block_end   = block_end if block_end > 0 else len(meta)
        block       = meta[block_start:block_end]
        headers = _re.findall(r'_struct_asym\.(\S+)', block)
        try:
            id_col  = headers.index('id')
            eid_col = headers.index('entity_id')
        except ValueError:
            search_pos = sa_idx + 1
            continue
        last_hdr   = f'_struct_asym.{headers[-1]}'
        data_start = block.find(last_hdr) + len(last_hdr)
        for line in block[data_start:].split('\n'):
            parts = line.split()
            if len(parts) > max(id_col, eid_col) and parts[eid_col] in trna_eids:
                label_ids.add(parts[id_col])
        search_pos = sa_idx + 1
    print(f'  tRNA label_asym_ids: {label_ids}')
    if not label_ids:
        return set()

    # ── Step 3: map label_asym_id -> auth_asym_id via _atom_site ─────────────
    # Parse column indices from the atom_site header block
    atom_start = cif_text.find('\nloop_\n_atom_site.')
    if atom_start < 0:
        atom_start = cif_text.find('\n_atom_site.')
    hdr_block  = cif_text[atom_start:atom_start + 5000]
    hdr_names  = _re.findall(r'_atom_site\.(\S+)', hdr_block)
    try:
        label_col = hdr_names.index('label_asym_id')
        auth_col  = hdr_names.index('auth_asym_id')
    except ValueError:
        print('  WARNING: could not find label_asym_id/auth_asym_id columns; returning label IDs')
        return label_ids  # fallback

    last_hdr_str = f'_atom_site.{hdr_names[-1]}'
    data_off = cif_text.find(last_hdr_str, atom_start) + len(last_hdr_str)
    label_to_auth = {}
    remaining = label_ids.copy()
    for line in cif_text[data_off:].split('\n'):
        if not remaining:
            break
        parts = line.split()
        if len(parts) <= max(label_col, auth_col):
            continue
        lbl = parts[label_col]
        if lbl in remaining:
            label_to_auth[lbl] = parts[auth_col]
            remaining.discard(lbl)

    auth_ids = set(label_to_auth.values())
    print(f'  tRNA auth_asym_ids (for py3Dmol): {sorted(auth_ids)}')
    return auth_ids

trna_chains = _get_trna_chains(cif_str)
print(f'Detected tRNA chains: {sorted(trna_chains) if trna_chains else "none found"}')


  tRNA entity IDs: {'36', '35'}
  tRNA label_asym_ids: {'IA', 'JA'}
  tRNA auth_asym_ids (for py3Dmol): ['s', 't']
Detected tRNA chains: ['s', 't']


In [13]:
view1 = py3Dmol.view(width=950, height=620)
view1.addModel(cif_str, 'cif')

# Base cartoon for ALL atoms — rRNA shows as grey ribbon
RNAcolor = "#888888"
view1.setStyle({}, {'cartoon': {'color': RNAcolor, 'opacity': 0.4}})

# Strip cartoon from protein chains only — tRNA chains are excluded here
# so their cartoon is never removed and will be re-styled as black below
all_prot_chains = {cm['chain_id'] for uid in sifts for cm in sifts[uid]} - trna_chains
for chain in all_prot_chains:
    view1.setStyle({'chain': chain}, {})

# ── Determine background grey per chain (RPL = dark, RPS = light) ────────────
# Build UniProt -> gene name from annotprot (handles comma-separated proteins)
uniprot_to_gene = {}
for _, row in annotprot.iterrows():
    gene = str(row['gene_name']).upper()
    for uid in _proteins_of(row['Protein']):
        if uid not in uniprot_to_gene:
            uniprot_to_gene[uid] = gene

def _chain_grey_color(uid):
    """Return the background grey for a protein chain.

    Priority:
    1. Gene name from annotprot (RPL* / RPS*)
    2. SIFTS UniProt identifier prefix (RL* -> RPL, RS*/GBLP -> RPS)
    3. COLOR_UNDETECTED for anything else (e.g. co-crystallised non-ribosomal proteins)
    """
    gene = uniprot_to_gene.get(uid, '').upper()
    if gene.startswith('RPL'):
        return COLOR_RPL
    if gene.startswith('RPS'):
        return COLOR_RPS
    # Fall back to SIFTS identifier (catches proteins not detected by LiP-MS
    # whose gene name is absent from annotprot, e.g. RACK1/GBLP, RSSA1)
    ident_pfx = sifts_ident.get(uid, '').split('_')[0].upper()
    if ident_pfx.startswith('RL'):
        return COLOR_RPL
    if ident_pfx.startswith('RS') or ident_pfx in ('GBLP', 'RSSA'):
        return COLOR_RPS
    return COLOR_UNDETECTED

chain_grey   = {}
unclassified = []
for uid, chain_maps in sifts.items():
    grey = _chain_grey_color(uid)
    if grey == COLOR_UNDETECTED:
        unclassified.append((uid, sifts_ident.get(uid, '')))
    for cm in chain_maps:
        chain_grey[cm['chain_id']] = grey

if unclassified:
    print(f'Chains shown in mid-grey (not RPL/RPS): {len(unclassified)}')
    for uid, ident in unclassified:
        print(f'  {uid}  {ident}')

# ── Group significant/correlating residues by chain ──────────────────────────
corr_by_chain = defaultdict(set)
for chain, res in corr_res_set:
    corr_by_chain[chain].add(res)

sig_only_by_chain = defaultdict(set)
for chain, res in sig_res_set:
    if res not in corr_by_chain[chain]:
        sig_only_by_chain[chain].add(res)

# ── Build per-chain full residue sets from ALL SIFTS entries ─────────────────
# Used to exclude colored residues from the grey background surface so that
# green/orange regions show their pure color without grey blending through.
chain_all_residues_sifts = defaultdict(set)
for uid, chain_maps in sifts.items():
    for cm in chain_maps:
        offset = cm['pdb_start'] - cm['unp_start']
        for unp_res in range(cm['unp_start'], cm['unp_end'] + 1):
            chain_all_residues_sifts[cm['chain_id']].add(unp_res + offset)

# ── Render protein surfaces ───────────────────────────────────────────────────
# Grey surface is drawn only on residues NOT covered by colored peptides,
# so green/orange regions have no grey surface underneath — pure colors only.
for chain in all_prot_chains:
    grey = chain_grey.get(chain, COLOR_UNDETECTED)
    colored = sig_only_by_chain[chain] | corr_by_chain[chain]

    if colored and chain in chain_all_residues_sifts:
        # Grey only for residues NOT covered by significant/correlating peptides
        grey_residues = sorted(chain_all_residues_sifts[chain] - colored)
        if grey_residues:
            view1.addSurface(py3Dmol.VDW,
                             {'opacity': 0.8, 'color': grey},
                             {'chain': chain, 'resi': grey_residues})
    else:
        # No colored residues on this chain — full chain grey surface
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 0.8, 'color': grey},
                         {'chain': chain})

    # Significant non-correlating residues in pure green (no grey underneath)
    if sig_only_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_SIG},
                         {'chain': chain, 'resi': sorted(sig_only_by_chain[chain])})

    # Correlating residues in pure orange-red (no grey underneath)
    if corr_by_chain[chain]:
        view1.addSurface(py3Dmol.VDW,
                         {'opacity': 1.0, 'color': COLOR_CORR},
                         {'chain': chain, 'resi': sorted(corr_by_chain[chain])})

# tRNA chains — apply black cartoon LAST so it is never stripped by the protein
# loop above (applied after all setStyle({chain: X}, {}) calls)
for chain in trna_chains:
    view1.setStyle({'chain': chain}, {'cartoon': {'color': 'black', 'opacity': 1.0}})

view1.zoomTo()
view1.show()

Chains shown in mid-grey (not RPL/RPS): 1
  P68105  EF1A1_RABIT


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [14]:
os.makedirs('plots', exist_ok=True)
out_path1 = f'plots/{OUTPUT_PREFIX}_peptide_level.html'
view1.write_html(out_path1)
print(f'Saved: {out_path1}')

Saved: plots/ribosome_peptide_level.html


## 8. Summary Table

In [15]:
# Build a summary of all mapped ribosomal peptides with their classification
rows = []
for pept_id in siglip_ribo:
    row = annotprot.loc[pept_id]
    category = 'correlating' if pept_id in sig_and_corr_ribo else 'sig_only'
    color    = COLOR_CORR if category == 'correlating' else COLOR_SIG
    mapped   = pept_id in pept_to_pdb
    chains   = '; '.join({pos['chain'] for pos in pept_to_pdb.get(pept_id, [])}) if mapped else ''
    rows.append({
        'peptide':    pept_id,
        'protein':    row['Protein'],
        'gene':       row['gene_name'],
        'start':      row['startPosition'],
        'end':        row['endPosition'],
        'category':   category,
        'color':      color,
        'mapped':     mapped,
        'pdb_chains': chains,
    })

summary_df = pd.DataFrame(rows).sort_values(['gene', 'start'])
print(f'Total matched-protein LiP-MS peptides:  {len(summary_df)}')
print(f'Successfully mapped to structure:        {summary_df.mapped.sum()}')
display(summary_df.head(20))

os.makedirs('source_data', exist_ok=True)
out_tsv = f'source_data/{OUTPUT_PREFIX}_lipms_mapping.tsv'
summary_df.to_csv(out_tsv, sep='\t', index=False)
print(f'\nSaved: {out_tsv}')

Total matched-protein LiP-MS peptides:  127
Successfully mapped to structure:        123


,peptide,protein,gene,start,end,category,color,mapped,pdb_chains
35,GTLEGHNGWVTSLATSAGQPNLLLSASR,P38011,ASC1,11,38,sig_only,#ea9c8e,True,SO
89,ASMIISGSR,P38011,ASC1,119,127,sig_only,#ea9c8e,True,SO
57,AAEPHAVSL,P38011,ASC1,284,292,correlating,#d8624d,True,SO
58,VWQVMTAN,P38011,ASC1,312,319,sig_only,#ea9c8e,True,SO
110,LLLSASR,P38011,ASC1,32,38,sig_only,#ea9c8e,True,SO
73,ALSASWDK,P38011,ASC1,80,87,sig_only,#ea9c8e,True,SO
50,LWDVATGETYQR,P38011,ASC1,91,102,correlating,#d8624d,True,SO
116,MLSCAGADR,P41805,RPL10,102,110,sig_only,#ea9c8e,True,L
82,TKDSNKDVVVEGLR,P41805,RPL10,140,153,correlating,#d8624d,True,L
74,DSNKDVVVEGLR,P41805,RPL10,142,153,sig_only,#ea9c8e,True,L



Saved: source_data/ribosome_lipms_mapping.tsv
